In [1]:
%load_ext autoreload
%autoreload 2

In [3]:
import json
from pathlib import Path
import pandas as pd
import requests
from ift6758.llm.get_better import (
    extract_text,
    load_function,
    format_test_report,
    build_refine_prompt,
    all_passed,
)
from ift6758.llm.chat import confirm, ask_user
from ift6758.llm.prompts import PROFILES, OG_TACHE_1
from ift6758.llm.qwen_log import create_log, log_call
from ift6758.llm.qwen_utils import (
    load_qwen,
    build_messages,
    generate,
    ask_qwen,
    check_env,
)
from ift6758.llm.rag import (
    load_doc,
    extract_section,
    split_doc,
    load_embedder,
    embed,
    retrieve,
    format_context,
    DOCS_DIR,
)
from ift6758.llm.test_both_code import run_tests, run_season_tests
from ift6758.llm.test_tidy_llm import run_tidy_tests

check_env()  # Demo 2, for reproductibility

Python: 3.11.15
NumPy: 1.26.4 | pandas: 2.3.3
Transformers: 5.17.0
Sentence Transformers: 6.1.0
CUDA GPU: Unavailable — CPU/Apple GPU fallback


In [4]:
# Code adapté de la démo 2
GAME_ID = 2022030411
raw_path = Path(DOCS_DIR.parent / "raw") / f"{GAME_ID}.json"
print("Cached game:", raw_path.resolve())
if not raw_path.exists():
    response = requests.get(
        f"https://api-web.nhle.com/v1/gamecenter/{GAME_ID}/play-by-play", timeout=30
    )
    response.raise_for_status()
    downloaded = response.json()
    assert downloaded["id"] == GAME_ID and isinstance(downloaded["plays"], list)
    raw_path.parent.mkdir(parents=True, exist_ok=True)
    raw_path.write_text(response.text, encoding="utf-8")
game = json.loads(
    raw_path.read_text(encoding="utf-8")
)  # Convertie de json vers objet python
assert game["id"] == GAME_ID and isinstance(game["plays"], list)
print(game["awayTeam"]["abbrev"], "at", game["homeTeam"]["abbrev"], game["gameDate"])

Cached game: /Users/jcforget/Maitrise/Cours/IFT 6758/Projet/Github/NHL-data-analysis/ift6758/data/raw/2022030411.json
FLA at VGK 2023-06-03


Objectif: Créer une plus petite version du json (car il est énorme, trop gros pour un bon contexte pour le petit modèle). Ca remplace le RAG de la question 1.

plays = [55(tir),87(but),1001(but filet désert)] if p["evenId"] in game
homeTeam, awayTeam
rosterSpots = nom de marqueur, tireur, gardiens
context = "Sample of one real game JSON (json number), structure is identical for all game." + json.dumps(extrait, indent=2)
extra = "context_source":"Game sample num_game, events 55/87/1001

In [5]:
# context : replace Q1 RAG by sample from JSON.
SAMPLE_EVENTS = (55, 87, 1001)  # shot, goal (1451), empty net goal (0551)
sample_file = f"{GAME_ID}.json"

#Match : juste as peek_game_level
game_level = {
    "id": game["id"],
    "season": game["season"],
    "gameType": game["gameType"],
    "homeTeam": {"id": game["homeTeam"]["id"], "abbrev": game["homeTeam"]["abbrev"]},
    "awayTeam": {"id": game["awayTeam"]["id"], "abbrev": game["awayTeam"]["abbrev"]},
}

# Chosen play and corresponding position in game["plays"]
sample_plays = [(i, p) for i, p in enumerate(game["plays"]) if p["eventId"] in SAMPLE_EVENTS]

# peek_shooter_keys ("PlayerId" in k) + goalie
player_ids = {
    v for _, p in sample_plays for k, v in p["details"].items()
    if "PlayerId" in k or k == "goalieInNetId"
}

#peek_player_name
roster = [
    {key: r[key] for key in ("playerId", "teamId", "firstName", "lastName")}
    for r in game["rosterSpots"] if r["playerId"] in player_ids
]

# Can reuse retrieve()by format_context() like in Q1
passages = [("game level", game_level)]
passages += [(f"plays[{i}]", p) for i, p in sample_plays]
passages += [("rosterSpots (players of the plays above)", roster)]
results = [
    {"rank": rank, "index": label, "score": None, "text": json.dumps(part, indent=2)}
    for rank, (label, part) in enumerate(passages, start=1)
]
context = (
    "Sample of one real NHL play-by-play game JSON. All games have the same structure.\n\n"
    + format_context(results, source=sample_file)
)
print(context)
print(len(context), "characters |", len(player_ids), "players")

Sample of one real NHL play-by-play game JSON. All games have the same structure.

[1] (source:2022030411.json, passage game level) {
  "id": 2022030411,
  "season": 20222023,
  "gameType": 3,
  "homeTeam": {
    "id": 54,
    "abbrev": "VGK"
  },
  "awayTeam": {
    "id": 13,
    "abbrev": "FLA"
  }
}

[2] (source:2022030411.json, passage plays[12]) {
  "eventId": 55,
  "periodDescriptor": {
    "number": 1,
    "periodType": "REG",
    "maxRegulationPeriods": 3
  },
  "timeInPeriod": "01:29",
  "timeRemaining": "18:31",
  "situationCode": "1551",
  "homeTeamDefendingSide": "right",
  "typeCode": 506,
  "typeDescKey": "shot-on-goal",
  "sortOrder": 34,
  "details": {
    "xCoord": 73,
    "yCoord": 27,
    "zoneCode": "O",
    "shotType": "wrist",
    "shootingPlayerId": 8479372,
    "goalieInNetId": 8478499,
    "eventOwnerTeamId": 13,
    "awaySOG": 1,
    "homeSOG": 0
  }
}

[3] (source:2022030411.json, passage plays[78]) {
  "eventId": 87,
  "periodDescriptor": {
    "number": 1,


Je me suis fait avoir. Le joeur change de nom selon l'event:Un tir a details.shootingPlayerId, mais un but a details.scoringPlayerId (pas de shootingPlayerId). Ensuite, s'il y a un but désert, le but n'a pas de clé goalieInNetId.

In [6]:
tidy_task = """
Write a Python function json2tidy_llm(raw_data).
Goal: Convert one NHL play-by-play game (JSON) into a tidy pandas DataFrame, one row per event

Input:
- json2tidy_llm(raw_data) accepts either a str (path to a game JSON file, read as UTF-8) or a dict (one game, already loaded).
- The context shows a sample of the real JSON structure. All games have this structure.

Definitions (from the NHL play-by-play JSON):
- Game level: id = game ID, season (e.g. 20222023), gameType (2 = regular season, 3 = playoffs), homeTeam.id, awayTeam.id.
- Events are in plays. Keep only plays whose typeDescKey is "shot-on-goal" or "goal".
- The shooter is details.shootingPlayerId for a shot, details.scoringPlayerId for a goal.
- Player names are in rosterSpots: playerId -> firstName["default"] + " " + lastName["default"].
- details.goalieInNetId is absent when the net is empty.
- situationCode has 4 digits: away goalie, away skaters, home skaters, home goalie. Example: "1451" = away team has 4 skaters, home team has 5.

Output columns (strictly use these names):
- game_id, event_id, season, game_type
- period (periodDescriptor.number), time (timeInPeriod, "MM:SS")
- team_id (details.eventOwnerTeamId, the team that took the shot)
- event_type ("shot-on-goal" or "goal")
- x_coord, y_coord
- shooter (full name), goalie (full name of goalieInNetId)
- shot_type (details.shotType)
- empty_net (True if goalieInNetId is absent, else False)
- strength: for goals only, compare the skaters of the scoring team with the skaters of the other team:
    more = "power play", fewer = "shorthanded", equal = "even". For shots, strength is missing.

Requirements:
- Use pandas and json. Include imports and a docstring.
- One row per event: (game_id, event_id) must be unique.
- Keep unavailable values as missing (NaN or None). Never replace them with 0, False or "".
- Return a pandas DataFrame.
Return only json2tidy_llm() and its imports in a Python code block.
"""

In [7]:
model, tokenizer, device = load_qwen()
log_path = create_log()
print("Log :", log_path)


# Code suggested by Claude to embelish presentation
def banner(title):
    print(f"\n{'=' * 15} {title} {'=' * 15}")

Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

Log : /Users/jcforget/Maitrise/Cours/IFT 6758/Projet/Github/NHL-data-analysis/logs/chat_2026-10-17_10-07-50.jsonl


In [8]:
MAX_ITER = 5  # SafeGuard
task = tidy_task
task_version = "tidy_v0"  # first version (demo 2 ift6758)
iteration = 0
history = []  # Loop resume, seen at the end

try:
    while True:
        # Code genration section
        banner(f"Step {iteration} : code generation ({task_version})")
        print(task)
        gen_result = ask_qwen(
            model=model,
            tokenizer=tokenizer,
            log_path=log_path,
            prompt=task,
            context=context,
            profile=PROFILES["gen_tidy"],
            extra={
                "iteration": iteration,
                "task_version": task_version,
                "retrieved": results,
            },
        )

        # Extraction and humain in the lopp validation
        try:
            tidy_code = extract_text(gen_result["answer"], tag="code")
        except ValueError as e:
            tidy_code = gen_result["answer"]  # no code generatde
            test_results = {"0. Extraction code": f"ERROR : {e}"}
        else:
            banner("Extracted code")
            print(tidy_code)
            if not confirm("Execute and test code ?"):
                print("Stawp : code ist nicht gut.")
                break
            # Transforme
            try:
                fn = load_function(tidy_code, "json2tidy_llm")
            except Exception as e:  # SyntaxError, import manquant...
                test_results = {"0. Code loading": f"ERROR : {type(e).__name__}: {e}"}
            else:
                test_results = run_tidy_tests(fn)
        report = format_test_report(test_results)
        banner("Test results")
        print(report)
        history.append(
            {
                "iteration": iteration,
                "task_version": task_version,
                "passed": all_passed(test_results),
                "report": report,
            }
        )
        if all_passed(test_results):
            print("\nAll test are Ok !")
            break
        if iteration >= MAX_ITER:
            print(f"\n MAX_ITER ({MAX_ITER}) too many iteration")
            break
        # Prompt refinement
        banner(f"Step {iteration} : prompt refinement")
        refine_prompt = build_refine_prompt(
            task=task,
            code=tidy_code,
            context=context,
            report=report
        )  # Raffine le prompt
        refine_result = ask_qwen(
            model=model,
            tokenizer=tokenizer,
            log_path=log_path,
            prompt=refine_prompt,
            context=None,
            profile=PROFILES["refine"],
            extra={
                "iteration": iteration,
                "task_version": task_version,
                "test_results": test_results,
            },
        )
        try:
            new_task = extract_text(
                answer=refine_result["answer"], tag="new_prompt"
            ).strip()
        except ValueError as e:
            print(f"Stawp!!! : {e}")
            break
        banner("New prompt")
        print(new_task)
        if not confirm("Accept new prompt and generate new code ?"):
            print("Stawp : prompt ist nicht gut.")
            break
        # Netx task
        task = new_task
        iteration += 1
        task_version = f"tidy_v{iteration}"
except KeyboardInterrupt:
    print(f"\n Stopped at step {iteration}.")
pd.DataFrame(history)


=============== Step 0 : code generation (tidy_v0) ===============

Write a Python function json2tidy_llm(raw_data).
Goal: Convert one NHL play-by-play game (JSON) into a tidy pandas DataFrame, one row per event

Input:
- json2tidy_llm(raw_data) accepts either a str (path to a game JSON file, read as UTF-8) or a dict (one game, already loaded).
- The context shows a sample of the real JSON structure. All games have this structure.

Definitions (from the NHL play-by-play JSON):
- Game level: id = game ID, season (e.g. 20222023), gameType (2 = regular season, 3 = playoffs), homeTeam.id, awayTeam.id.
- Events are in plays. Keep only plays whose typeDescKey is "shot-on-goal" or "goal".
- The shooter is details.shootingPlayerId for a shot, details.scoringPlayerId for a goal.
- Player names are in rosterSpots: playerId -> firstName["default"] + " " + lastName["default"].
- details.goalieInNetId is absent when the net is empty.
- situationCode has 4 digits: away goalie, away skaters, home sk

,iteration,task_version,passed,report
0,0,tidy_v0,True,8/8 tests passed\nTest: 1. Returns DataFrame w...
